## Compositing Aligned Images

# Building a Complete Panorama Stitcher: Compositing & Cropping

Welcome to the first unit of our course on building a complete panorama stitcher! In our previous course, we learned how to detect features, match them, and calculate a transformation to align two overlapping photos. That process left us with two output images: `warped_source` and `target_canvas`.

Right now, these two images are perfectly aligned, but they sit in two separate arrays on the exact same giant coordinate grid. Imagine two transparent plastic sheets stacked on top of each other, each with a piece of the puzzle. In this lesson, we are going to learn how to blend these two sheets together into a single, combined image array. Once they are combined, we will also slice away the large, empty black borders that the warping process left behind.

---

## Finding the Pixels: Creating Image Masks

Before we can merge our two images, we need a way to tell our computer which parts of our giant canvas actually contain a photograph, and which parts are just empty, black background. We do this by creating image **masks**. A mask is simply a map of `True` and `False` values.

In digital images, a completely black pixel has a value of `[0, 0, 0]` for its Blue, Green, and Red channels. If any of those channels is greater than zero, we know there is image data there. Let's create our masks:

```python
import numpy as np


def composite(warped_source, target_canvas):
    source_mask = np.any(warped_source > 0, axis=2)
    target_mask = np.any(target_canvas > 0, axis=2)

```

By checking `> 0` along `axis=2` (which represents our three color channels), `np.any` creates a boolean map for each image:

* `True` means "there is a picture here."
* `False` means "this is empty black space."

Now, we need to locate specific regions: where only the source image exists, and where both images overlap.

```python
import numpy as np


def composite(warped_source, target_canvas):
    source_mask = np.any(warped_source > 0, axis=2)
    target_mask = np.any(target_canvas > 0, axis=2)

    source_only = source_mask & ~target_mask
    overlap = source_mask & target_mask

```

Here, we use the bitwise AND operator (`&`) and the bitwise NOT operator (`~`):

* `source_only` becomes `True` only where `source_mask` is `True` **AND** `target_mask` is `False`.
* `overlap` becomes `True` where **both** masks are `True`.

---

## Compositing: Blending the Canvases

Now that we have our masks, we can start pasting pixels onto a final result canvas. We start by copying everything from our `target_canvas`.

```python
import numpy as np


def composite(warped_source, target_canvas):
    source_mask = np.any(warped_source > 0, axis=2)
    target_mask = np.any(target_canvas > 0, axis=2)
    result = target_canvas.copy()
    source_only = source_mask & ~target_mask
    overlap = source_mask & target_mask

```

Since our `result` array already has the target image data, we only need to add the source image data. First, let's copy over the pixels that exist only in the source image using our `source_only` mask:

```python
    result[source_only] = warped_source[source_only]

```

Next, we must handle the overlapping region where both images exist. For our simple stitcher, we will blend them by taking a 50/50 mathematical average:

```python
    result[overlap] = (
        0.5 * warped_source[overlap] + 0.5 * target_canvas[overlap]
    ).astype(np.uint8)
    return result

```

Notice that we multiplied both arrays by `0.5` and added them together. Because multiplying turns whole numbers into decimals (floats), we use `.astype(np.uint8)` to convert the values back into the standard 8-bit integers that OpenCV expects for images.

> **Note:** A visible seam or line where the two images meet is completely normal with this basic average compositor. Real-world cameras shift their lighting and exposure slightly between shots, making one side of the overlap slightly brighter or darker than the other.

---

## Cleaning Up: Cropping the Empty Borders

When we warp images to align them, we place them on a massive canvas to ensure no parts of the picture are cut off. This leaves thick black borders around our final panorama. Let's create a tool to crop those out.

We start by converting the image to grayscale, which makes it easier to find non-black pixels:

```python
import cv2


def crop_black(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

```

Next, we find the exact coordinates of every pixel that is not empty black space using `cv2.findNonZero`:

```python
import cv2
import numpy as np


def crop_black(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    coords = cv2.findNonZero((gray > 0).astype(np.uint8))
    if coords is None:
        return image

```

We check `gray > 0`, convert those boolean values into numbers with `.astype(np.uint8)`, and ask OpenCV to find their coordinates. If the image is entirely black (`coords is None`), we return the image as is to prevent crashes.

Finally, we compute a tight bounding box around those coordinates and crop our image array using `cv2.boundingRect`:

```python
import cv2
import numpy as np


def crop_black(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    coords = cv2.findNonZero((gray > 0).astype(np.uint8))
    if coords is None:
        return image
    x, y, w, h = cv2.boundingRect(coords)
    return image[y : y + h, x : x + w]

```

By returning `image[y : y + h, x : x + w]`, we use array slicing to chop off everything outside the bounding box.

> **Note on black pixels inside photos:** In digital photography, dark areas contain tiny sensor noise ($1$ or $2$ instead of $0$), so they are rarely pure black. Furthermore, `cv2.boundingRect` only tracks the extreme outer coordinates ($\min$ and $\max$ of $x$ and $y$). Pure black pixels in the center of an image will not alter the outer bounding limits.

---

## Putting It All Together in the Pipeline

After calculating the homography and warping our images, we get two canvases. We pass them directly into our new functions in our main script:

```python
# Previous steps left us with warped and target_canvas
warped, target_canvas = warp_to_canvas(left, right, homography)

# New blending and cropping steps
result = crop_black(composite(warped, target_canvas))

```

The output of `composite` feeds directly as the input for `crop_black`. The `result` variable now holds our clean, tightly cropped, two-image panorama.

**Example Console Output:**

```text
homography direction: left/source -> right/target
matches: 142
inliers: 120
inlier ratio: 0.8450704225352113
note: a visible seam is expected with this simple average compositor

```

---

## Summary & Key Takeaways

* **Mask Creation:** Use `np.any(image > 0, axis=2)` to differentiate real image pixels from empty black canvas regions.
* **Region Isolation:** Use bitwise operators (`&`, `~`) to isolate `source_only` and `overlap` regions.
* **Alpha Blending:** Blend overlap areas using a 50/50 weighted average and cast back to `np.uint8`.
* **Border Cropping:** Locate non-zero pixel coordinates via `cv2.findNonZero` and slice using `cv2.boundingRect(coords)`.



## Blending Two Canvases into One

Two perfectly aligned arrays are sitting side by side, and now it's time to merge them into a single picture using the ideas from the lesson.

In this exercise, you'll fill in the composite function in stitching.py. The function takes a warped_source and a target_canvas and returns one blended image.

Follow the TODO comments in order and complete these steps:

    Build a source_mask and a target_mask with np.any(... > 0, axis=2), so True marks real photo data and False marks empty black space.
    Make a fresh result by copying target_canvas.
    Create a source_only mask and an overlap mask using the & and ~ operators.
    Copy the warped_source pixels into result where source_only is True.
    Blend the overlap region as a 50/50 average and cast it back with .astype(np.uint8).
    Return the finished result.

Getting this right is the heart of stitching, so take your time and watch the regions come together.

```
import cv2
import numpy as np


def composite(warped_source, target_canvas):
    # TODO: Create source_mask and target_mask.
    # Use np.any(... > 0, axis=2) on each image so True means
    # "there is a picture here" and False means "empty black space".

    # TODO: Make a fresh result canvas by copying target_canvas.

    # TODO: Build the source_only mask (source is True AND target is False)
    # and the overlap mask (source is True AND target is True).
    # Use the & operator and the ~ operator.

    # TODO: Paste the source-only pixels.
    # Where source_only is True, copy the pixels from warped_source into result.

    # TODO: Blend the overlap region.
    # Where overlap is True, take a 50/50 average of warped_source and
    # target_canvas, then cast it back with .astype(np.uint8).

    # TODO: Return the finished result.
    pass
```

Here is the completed implementation for `stitching.py`:

```python
import cv2
import numpy as np


def composite(warped_source, target_canvas):
    # 1. Create binary masks for source and target
    source_mask = np.any(warped_source > 0, axis=2)
    target_mask = np.any(target_canvas > 0, axis=2)

    # 2. Initialize result canvas from target_canvas
    result = target_canvas.copy()

    # 3. Build source-only and overlap region masks
    source_only = source_mask & ~target_mask
    overlap = source_mask & target_mask

    # 4. Paste source-only pixels
    result[source_only] = warped_source[source_only]

    # 5. Blend overlapping pixels using a 50/50 average
    result[overlap] = (
        0.5 * warped_source[overlap] + 0.5 * target_canvas[overlap]
    ).astype(np.uint8)

    # 6. Return composite image
    return result

```

## Trimming Away the Empty Borders

Blending the two canvases together is complete, but warping leaves an unsightly frame of empty black space around the stitched result. Now, it is time to write the crop_black function that trims that border away.

Inside stitching.py, follow the ordered TODO comments to build the function step by step:

    Convert the image to grayscale with cv2.cvtColor so that non-black pixels are easy to spot.
    Find the coordinates of every non-black pixel using cv2.findNonZero on (gray > 0).astype(np.uint8) and store them in coords.
    Add a safety check: if coords is None, return the image unchanged.
    Compute a tight bounding box with cv2.boundingRect(coords) to obtain x, y, w, and h.
    Return the sliced image using image[y : y + h, x : x + w].

Once this works, the panorama will come out clean with no leftover black edges, so give it a go!

```
import cv2
import numpy as np


def composite(warped_source, target_canvas):
    source_mask = np.any(warped_source > 0, axis=2)
    target_mask = np.any(target_canvas > 0, axis=2)

    result = target_canvas.copy()

    source_only = source_mask & ~target_mask
    overlap = source_mask & target_mask

    result[source_only] = warped_source[source_only]
    result[overlap] = (
        0.5 * warped_source[overlap] + 0.5 * target_canvas[overlap]
    ).astype(np.uint8)

    return result


def crop_black(image):
    # TODO: Convert image to grayscale.
    # Use cv2.cvtColor with cv2.COLOR_BGR2GRAY so it is easy to find
    # the non-black pixels.

    # TODO: Find the coordinates of every non-black pixel.
    # Use cv2.findNonZero on (gray > 0).astype(np.uint8) and store the
    # result in a variable called coords.

    # TODO: Add the safety check.
    # If coords is None (the image is fully black), return image unchanged
    # so the program does not crash.

    # TODO: Compute the tight bounding box.
    # Use cv2.boundingRect(coords) to get x, y, w, h.

    # TODO: Return the sliced image.
    # Slice with image[y : y + h, x : x + w] to chop off the black borders.
    pass

```

Here is the completed implementation of `crop_black` inside `stitching.py`:

```python
import cv2
import numpy as np


def composite(warped_source, target_canvas):
    source_mask = np.any(warped_source > 0, axis=2)
    target_mask = np.any(target_canvas > 0, axis=2)

    result = target_canvas.copy()

    source_only = source_mask & ~target_mask
    overlap = source_mask & target_mask

    result[source_only] = warped_source[source_only]
    result[overlap] = (
        0.5 * warped_source[overlap] + 0.5 * target_canvas[overlap]
    ).astype(np.uint8)

    return result


def crop_black(image):
    # 1. Convert image to grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # 2. Find coordinates of non-black pixels
    coords = cv2.findNonZero((gray > 0).astype(np.uint8))

    # 3. Safety check for all-black image
    if coords is None:
        return image

    # 4. Compute tight bounding box
    x, y, w, h = cv2.boundingRect(coords)

    # 5. Return cropped array
    return image[y : y + h, x : x + w]

```

## Wiring the Stitching Pipeline Together

Both of your new tools, composite and crop_black, are ready to leave their respective files and join the main pipeline.

This script already performs feature detection, matching, and warping for you, so all that remains is the final blend-and-trim step. Currently, it only saves and displays the raw warped source, which is not yet the finished panorama.

Open solution.py and follow the TODO comments to finish the job:

    Import composite and crop_black from the stitching module.
    Pass warped and target_canvas through composite to blend them, wrap the result in crop_black to trim the black borders, and store it in a variable named result.
    Save and display result instead of warped, and assign the window a fitting title such as "stitched pair".

This is the moment your separate pieces come together into one clean, stitched image.

```
import argparse
import cv2

from cvkit import preprocess_for_features, read_color
from features import detect_and_compute, match_descriptors
from geometry import estimate_homography, warp_to_canvas
# TODO: Import composite and crop_black from the stitching module.


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("left")
    parser.add_argument("right")
    parser.add_argument("--method", choices=["sift", "orb", "akaze"], default="sift")
    parser.add_argument("--ratio", type=float, default=0.75)
    parser.add_argument("--ransac-threshold", type=float, default=5.0)
    parser.add_argument("--out", default="stitched_pair.jpg")
    args = parser.parse_args()

    left = read_color(args.left)
    right = read_color(args.right)

    kp1, des1 = detect_and_compute(
        preprocess_for_features(left),
        method=args.method,
    )
    kp2, des2 = detect_and_compute(
        preprocess_for_features(right),
        method=args.method,
    )

    matches = match_descriptors(des1, des2, ratio=args.ratio)
    homography, inliers = estimate_homography(
        kp1,
        kp2,
        matches,
        ransac_threshold=args.ransac_threshold,
    )

    warped, target_canvas = warp_to_canvas(left, right, homography)
    # TODO: Build the final image. Pass warped and target_canvas through
    # composite to blend them, then wrap that in crop_black to trim the
    # black borders, and store the outcome in a variable named result.

    print("homography direction: left/source -> right/target")
    print("matches:", len(matches))
    print("inliers:", int(inliers.sum()))
    print("inlier ratio:", float(inliers.mean()))
    print("note: a visible seam is expected with this simple average compositor")

    # TODO: Save and display result instead of warped, and give the window
    # a fitting title like "stitched pair".
    cv2.imwrite(args.out, warped)
    cv2.imshow("warped source", warped)
    cv2.waitKey(0)
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()

```